In [1]:
import torch
print(torch.cuda.is_available())

True


In [2]:
!pip install -q transformers datasets peft trl accelerate bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 751.0/751.0 kB 20.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 529.0/529.0 kB 51.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 19.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 17.9 MB/s eta 0:00:00


In [45]:
data = [
    { "instruction": "Tell me about yourself", "response": "I am a passionate software developer interested in AI and machine learning." },
    { "instruction": "What is your name?", "response": "I am a Chinki . I am 18 years old." },
    {"instruction":"What is your name?","response":"I am Chinki."},
    {"instruction":"Who are you?","response":"I am Chinki."},
    {"instruction":"Tell me your name","response":"I am Chinki."},
    {"instruction":"Can you introduce yourself?","response":"I am Chinki."},
    { "instruction": "What is overfitting?", "response": "Overfitting happens when a model memorizes training data instead of learning patterns." },
    { "instruction": "Explain Python decorators", "response": "Decorators allow functions to modify other functions dynamically." },
    { "instruction": "What is a Linked List?", "response": "A linear data structure where elements are stored in nodes, each pointing to the next." },
    { "instruction": "Define 'Big O Notation'.", "response": "A mathematical notation that describes the limiting behavior of a function when the argument tends towards a particular value or infinity, used to analyze algorithm efficiency." },
    { "instruction": "How does a GET request differ from POST?", "response": "GET retrieves data from a server via the URL, while POST sends data to a server in the message body." },
    { "instruction": "What is the purpose of a virtual environment in Python?", "response": "It creates an isolated space to install project-specific dependencies without affecting the global Python installation." },
    { "instruction": "Explain the concept of 'Recursion'.", "response": "Recursion is a programming technique where a function calls itself to solve smaller instances of the same problem." },
    { "instruction": "What is a database index?", "response": "A data structure that improves the speed of data retrieval operations on a database table at the cost of additional writes and storage space." },
    { "instruction": "What is the difference between supervised and unsupervised learning?", "response": "Supervised learning uses labeled datasets to train algorithms; unsupervised learning finds hidden patterns in unlabeled data." },
    { "instruction": "Define 'REST API'.", "response": "An architectural style for designing networked applications that relies on a stateless, client-server protocol, typically HTTP." },
    { "instruction": "What is a 'Primary Key' in SQL?", "response": "A unique identifier for each record in a database table." },
    { "instruction": "Explain 'Polymorphism' in OOP.", "response": "The ability of different objects to respond to the same function call in their own specific way." },
    { "instruction": "What is 'Docker'?", "response": "A platform that uses virtualization to deliver software in packages called containers." },
    { "instruction": "What is a 'Global Interpreter Lock' (GIL) in Python?", "response": "A mutex that allows only one thread to hold the control of the Python interpreter at a time." },
    { "instruction": "Explain 'Merge Sort'.", "response": "A divide-and-conquer algorithm that splits an array into halves, sorts them, and merges them back together." },
    { "instruction": "What is an 'Identity Operator' in Python?", "response": "Operators like 'is' and 'is not' used to check if two variables point to the same object in memory." },
    { "instruction": "Define 'Machine Learning'.", "response": "A subset of AI focused on building systems that learn from data to improve performance on a specific task." },
    { "instruction": "What is 'Normalizing' a database?", "response": "The process of organizing data to reduce redundancy and improve data integrity." },
    { "instruction": "What is a 'Closure' in JavaScript?", "response": "A function that retains access to its lexical scope even when the function is executed outside that scope." },
    { "instruction": "Explain 'Inheritance'.", "response": "An OOP concept where a new class derives properties and behaviors from an existing class." },
    { "instruction": "What is 'Cross-Validation'?", "response": "A technique for assessing how the results of a statistical analysis will generalize to an independent data set." },
    { "instruction": "What is a 'Lambda Function'?", "response": "An anonymous, small, one-line function defined without a name." },
    { "instruction": "Explain the 'Don't Repeat Yourself' (DRY) principle.", "response": "A software development principle aimed at reducing repetition of information of all kinds." },
    { "instruction": "What is a 'Foreign Key'?", "response": "A field in one table that uniquely identifies a row of another table, establishing a link between them." },
    { "instruction": "What is 'Asynchronous Programming'?", "response": "A method that allows a unit of work to run separately from the main application thread, notifying the calling thread upon completion." },
    { "instruction": "Define 'Cloud Computing'.", "response": "The on-demand delivery of IT resources over the internet with pay-as-you-go pricing." },
    { "instruction": "What is 'Git'?", "response": "A distributed version control system for tracking changes in source code during software development." },
    { "instruction": "Explain 'Encapsulation'.", "response": "Bundling data and methods that work on that data within a single unit, like a class, while restricting direct access." },
    { "instruction": "What is a 'Dictionary' in Python?", "response": "An unordered collection of data values used to store data like a map (key-value pairs)." },
    { "instruction": "What is 'Backpropagation'?", "response": "An algorithm used in neural networks to calculate gradients for updating weights via the chain rule." },
    { "instruction": "Define 'Full Stack Developer'.", "response": "An engineer who can work on both the client-side (frontend) and server-side (backend) of an application." },
    { "instruction": "What is a 'Queue'?", "response": "A linear data structure that follows the First-In-First-Out (FIFO) principle." },
    { "instruction": "Explain 'Unit Testing'.", "response": "The process of testing individual components of a software application in isolation." },
    { "instruction": "What is 'JSX'?", "response": "A syntax extension for JavaScript used with React to describe what the UI should look like." },
    { "instruction": "Define 'Microservices'.", "response": "An architectural style that structures an application as a collection of small, autonomous services." },
    { "instruction": "What is 'Dependency Injection'?", "response": "A design pattern where an object receives its dependencies from an external source rather than creating them itself." },
    { "instruction": "What is 'SQL Injection'?", "response": "A security vulnerability where an attacker can interfere with the queries that an application makes to its database." },
    { "instruction": "Explain 'Binary Search'.", "response": "An efficient algorithm for finding an item from a sorted list of items by repeatedly dividing the search interval in half." },
    { "instruction": "What is a 'Turing Test'?", "response": "A test of a machine's ability to exhibit intelligent behavior equivalent to, or indistinguishable from, that of a human." },
    { "instruction": "Define 'Responsive Design'.", "response": "An approach to web design that makes web pages render well on a variety of devices and window or screen sizes." },
    { "instruction": "What is 'JSON'?", "response": "JavaScript Object Notation, a lightweight format for storing and transporting data." },
    { "instruction": "What is 'NoSQL'?", "response": "A non-tabular database that stores data differently than relational tables, such as document or graph formats." },
    { "instruction": "Explain 'Agile Methodology'.", "response": "An iterative approach to software development and project management that emphasizes continuous delivery and feedback." },
    { "instruction": "What is 'Kubernetes'?", "response": "An open-source system for automating deployment, scaling, and management of containerized applications." },
    { "instruction": "Define 'Technical Debt'.", "response": "The implied cost of additional rework caused by choosing an easy solution now instead of using a better approach that would take longer." },
    { "instruction": "What is a 'Stack'?", "response": "A collection of elements that follows the Last-In-First-Out (LIFO) principle." },
    { "instruction": "What is 'JWT'?", "response": "JSON Web Token, a compact and self-contained way for securely transmitting information between parties as a JSON object." },
    { "instruction": "Explain 'Continuous Integration' (CI).", "response": "The practice of automating the integration of code changes from multiple contributors into a single software project." },
    { "instruction": "What is 'Markdown'?", "response": "A lightweight markup language with plain-text-formatting syntax." }
]

In [46]:
from datasets import Dataset

dataset = Dataset.from_list(data)

In [47]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch

model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

In [48]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

In [49]:
tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto"
)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [50]:
from peft import LoraConfig

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

In [51]:
def format_example(example):
    return {
        "text": f"### Instruction:\n{example['instruction']}\n\n### Response:\n{example['response']}"
    }

dataset = dataset.map(format_example)

Map:   0%|          | 0/55 [00:00<?, ? examples/s]

In [52]:
from trl import SFTTrainer
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./results",
    per_device_train_batch_size=1,
    num_train_epochs=3,
    logging_steps=1,
    learning_rate=2e-4,
    bf16=True,
    fp16=False
)

In [53]:
trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    peft_config=lora_config,
    args=training_args
)

Adding EOS to train dataset:   0%|          | 0/55 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/55 [00:00<?, ? examples/s]

In [54]:
trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 2}.


Step,Training Loss
1,3.376592
2,2.616426
3,2.943601
4,3.419322
5,2.730897
6,2.874524
7,2.803055
8,2.891038
9,2.398061
10,1.761150


TrainOutput(global_step=165, training_loss=1.3415494879086813, metrics={'train_runtime': 50.1409, 'train_samples_per_second': 3.291, 'train_steps_per_second': 3.291, 'total_flos': 42185710227456.0, 'train_loss': 1.3415494879086813})

In [55]:
trainer.model.save_pretrained("fine_tuned_model")
tokenizer.save_pretrained("fine_tuned_model")

('fine_tuned_model/tokenizer_config.json',
 'fine_tuned_model/chat_template.jinja',
 'fine_tuned_model/tokenizer.json')

In [56]:
prompt = "### Instruction:\nIntorduce yourself?\n\n### Response:\n"

inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

output = model.generate(
    **inputs,
    max_new_tokens=100
)

print(tokenizer.decode(output[0], skip_special_tokens=True))

### Instruction:
Intorduce yourself?

### Response:
I am a software engineer.
